# Evaluating Multiple LM Outputs (External)

In [3]:
# imports
import json
import pandas as pd
import importlib.util
import sys
from os.path import join
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.additions.eval.extraction import \
    format_features, format_model_info
from stat_genie.blade_pipeline.additions.analysis.conclusion import \
    write_final_answer_code, make_conclusion
from stat_genie.blade_pipeline.additions.eval.judge import run_judge_evaluation_pairwise

In [42]:
# load files
output_dir = "/accounts/projects/binyu/hao_huang/stat-genie/examples/output"
analysis_subdir_path_1 = f"{output_dir}/soccer_ana_true"
analysis_subdir_path_2 = f"{output_dir}/soccer_shuffle_ana_true"

# CHANGE THIS PATH TO WHERE YOU WANT TO SAVE THE EVALUATION RESULTS
output_save_path = f"{output_dir}/external_eval_results.json"
multirun_filename_1 = "multirun_analyses.json"
multirun_filename_2 = "multirun_analyses.json"

# use both files to get analysis code paths
multirun_path_1 = join(analysis_subdir_path_1, multirun_filename_1)
multirun_path_2 = join(analysis_subdir_path_2, multirun_filename_2)

with open(multirun_path_1, "r") as file:
    multirun_analyses_1 = json.load(file)

with open(multirun_path_2, "r") as file:
    multirun_analyses_2 = json.load(file)

num_analyses_1 = multirun_analyses_1['n']
num_analyses_2 = multirun_analyses_2['n']

analysis_code_filenames_1 = [f"llm_analysis_{i}.py" for i in range(num_analyses_1)]
analysis_code_filenames_2 = [f"llm_analysis_{i}.py" for i in range(num_analyses_2)]

analysis_code_paths_1 = [join(analysis_subdir_path_1, filename)
                         for filename in analysis_code_filenames_1]

analysis_code_paths_2 = [join(analysis_subdir_path_2, filename)
                         for filename in analysis_code_filenames_2]

In [43]:
llm_provider = "openai"
llm_model = "gpt-5-mini"
llm_assistant = llm(provider=llm_provider, model=llm_model)

[2025-12-05 02:16:53.34][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/projects/binyu/hao_huang/stat-genie/config/llm_eval_config.yml'.


In [44]:
features_1 = format_features(multirun_analyses_1, num_analyses_1, llm_assistant)

In [30]:
features_2 = format_features(multirun_analyses_2, num_analyses_2, llm_assistant)

[2025-12-05 01:56:04.67][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 01:56:10.83][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  6.16 seconds
[2025-12-05 01:56:10.83][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 01:56:10.84][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 01:56:18.70][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  7.85 seconds
[2025-12-05 01:56:18.70][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 01:56:18.71][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 01:56:25.63][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  6.92 

In [45]:
model_info_1 = format_model_info(multirun_analyses_1, num_analyses_1, llm_assistant)

In [31]:
model_info_2 = format_model_info(multirun_analyses_2, num_analyses_2, llm_assistant)

[2025-12-05 01:59:29.01][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 01:59:40.27][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  11.27 seconds
[2025-12-05 01:59:40.28][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 01:59:40.30][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 01:59:48.73][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  8.43 seconds
[2025-12-05 01:59:48.74][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 01:59:48.76][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 02:00:00.29][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  11.5

In [46]:
# load dataset, need more user-friendly input method later
dataset_name = multirun_analyses_1['dataset_name']
dataset_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                    "datasets", dataset_name, "data.csv")

data = pd.read_csv(dataset_path)

In [47]:
transform_functions_1 = {}
transform_functions_2 = {}
model_functions_1 = {}
model_functions_2 = {}

# ----- Loop for first set -----
for i, analysis_code_path in enumerate(analysis_code_paths_1):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_{i}"] = module
    spec.loader.exec_module(module)
    
    transform_functions_1[i] = module.transform
    model_functions_1[i] = module.model

# ----- Loop for second set -----
for i, analysis_code_path in enumerate(analysis_code_paths_2):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_2_{i}"] = module
    spec.loader.exec_module(module)

    transform_functions_2[i] = module.transform
    model_functions_2[i] = module.model

In [48]:
transformed_datasets_1 = {}
for i, transform_func in transform_functions_1.items():
    try:
        transformed_datasets_1[i] = transform_func(data.copy())
        print(f"[Transform 1-{i}] ✅ Completed successfully.")
    except Exception:
        print(f"[Transform 1-{i}] ❌")
        transformed_datasets_1[i] = None

model_results_1 = {}
for i, model_func in model_functions_1.items():
    try:
        if transformed_datasets_1[i] is None:
            print(f"[Model 1-{i}] ⚠️ Skipping — transform failed.")
            continue

        model_results_1[i] = model_func(transformed_datasets_1[i].copy())
        print(f"[Model 1-{i}] ✅ Completed successfully.")
    except Exception as e:
        print(f"[Model 1-{i}] ❌ Failed with error: {e}")
        model_results_1[i] = None

[Transform 1-0] ✅ Completed successfully.
[Transform 1-1] ✅ Completed successfully.
[Transform 1-2] ✅ Completed successfully.


/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 1-0] ❌ Failed with error: 'GLMResults' object has no attribute 'get_robustcov_results'


/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 1-1] ✅ Completed successfully.


/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 1-2] ✅ Completed successfully.


In [34]:
transformed_datasets_2 = {}
for i, transform_func in transform_functions_2.items():
    try:
        transformed_datasets_2[i] = transform_func(data.copy())
        print(f"[Transform 2-{i}] ✅ Completed successfully.")
    except Exception:
        print(f"[Transform 2-{i}] ❌")
        transformed_datasets_2[i] = None

model_results_2 = {}
for i, model_func in model_functions_2.items():
    try:
        if transformed_datasets_2[i] is None:
            print(f"[Model 2-{i}] ⚠️ Skipping — transform failed.")
            continue

        model_results_2[i] = model_func(transformed_datasets_2[i].copy())
        print(f"[Model 2-{i}] ✅ Completed successfully.")
    except Exception as e:
        print(f"[Model 2-{i}] ❌ Failed with error: {e}")
        model_results_2[i] = None

[Transform 2-0] ✅ Completed successfully.
[Transform 2-1] ✅ Completed successfully.
[Transform 2-2] ✅ Completed successfully.


/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 2-0] ✅ Completed successfully.


/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 2-1] ❌ Failed with error: 'GLMResults' object has no attribute 'get_robustcov_results'


/accounts/projects/binyu/hao_huang/stat-genie/.venv/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                 Generalized Linear Model Regression Results                  
Dep. Variable:               redCards   No. Observations:               107645
Model:                            GLM   Df Residuals:                   107626
Model Family:        NegativeBinomial   Df Model:                           18
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -6622.3
Date:                Fri, 05 Dec 2025   Deviance:                       9513.6
Time:                        14:00:43   Pearson chi2:                 1.08e+05
No. Iterations:                     7   Pseudo R-squ. (CS):           0.001670
Covariance Type:            nonrobust                                         
                                          coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [49]:
final_answer_code_1 = {}

info_json_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                      "datasets", dataset_name, "info.json")
with open(info_json_path, "r") as file:
    info_json = json.load(file)

task = info_json['research_questions']

for i in range(num_analyses_1):

    independent_variable = features_1[i]['independent_variables']
    dependent_variable = features_1[i]['response_variables']

    model_code = multirun_analyses_1['analyses'][str(i)]['m_code']
    model_output = model_results_1[i]

    final_answer_code_1[i] = write_final_answer_code(
        llm_assistant=llm_assistant,
        task=task,
        independent_variable=independent_variable,
        dependent_variable=dependent_variable,
        model_code=model_code,
        model_output=model_output,
    )

[2025-12-05 02:17:41.23][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)


[2025-12-05 02:18:20.05][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  38.82 seconds
[2025-12-05 02:18:20.06][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 02:18:20.08][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 02:18:47.55][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  27.47 seconds
[2025-12-05 02:18:47.55][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [35]:
final_answer_code_2 = {}
for i in range(num_analyses_2):

    independent_variable = features_2[i]['independent_variables']
    dependent_variable = features_2[i]['response_variables']

    model_code = multirun_analyses_2['analyses'][str(i)]['m_code']
    model_output = model_results_2[i]

    final_answer_code_2[i] = write_final_answer_code(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        model_output,
    )

[2025-12-05 02:00:43.35][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)


[2025-12-05 02:01:12.43][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  29.08 seconds
[2025-12-05 02:01:12.44][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 02:01:12.46][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 02:01:44.36][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  31.90 seconds
[2025-12-05 02:01:44.37][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 02:01:44.38][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 02:02:07.62][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  23.24 seconds
[2025-12-05 02:02:07.63][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)

In [36]:
print(final_answer_code_1)
print(final_answer_code_2)

{0: 'def extract_final_answer(model_output):\n    """\n    Extracts statistics for the DarkSkin coefficient from a fitted statsmodels GLM/GLMResultsWrapper.\n    Returns a dictionary with keys:\n      - "object": dict of extracted numeric results (coef, se, pval, CI, IRR, IRR_CI, percent change, significance, model info)\n      - "description": human-readable interpretation in the context of the research question.\n    """\n    import numpy as np\n    import pandas as pd\n\n    res = model_output\n\n    # Ensure results-like object\n    if res is None:\n        raise ValueError("model_output is None")\n\n    # Try to get parameter series\n    try:\n        params = res.params\n    except Exception:\n        raise ValueError("Provided model_output does not expose \'params\'")\n\n    # Find the parameter name corresponding to the DarkSkin variable\n    param_names = list(params.index)\n    candidates = [n for n in param_names if \'DarkSkin\' in str(n)]\n    if len(candidates) == 0:\n    

In [50]:
final_answer_functions_1 = {}

for i in range(num_analyses_1):
    namespace = {}

    compiled_code = compile(
        final_answer_code_1[i],
        f"<final_answer_code_1_{i}>",
        "exec"
    )
    exec(compiled_code, namespace)

    final_answer_functions_1[i] = namespace['extract_final_answer']

final_answers_1 = [
    final_answer_functions_1[i](model_results_1[i])
    for i in range(num_analyses_1)
]

In [37]:
final_answer_functions_2 = {}

for i in range(num_analyses_2):
    namespace = {}

    compiled_code = compile(
        final_answer_code_2[i],
        f"<final_answer_code_2_{i}>",
        "exec"
    )
    exec(compiled_code, namespace)

    final_answer_functions_2[i] = namespace['extract_final_answer']

final_answers_2 = [
    final_answer_functions_2[i](model_results_2[i])
    for i in range(num_analyses_2)
]

In [51]:
conclusions_1 = {}

for i in range(num_analyses_1):
    independent_variable = features_1[i]['independent_variables']
    dependent_variable = features_1[i]['response_variables']

    model_code = multirun_analyses_1['analyses'][str(i)]['m_code']
    interpretation_code = final_answer_code_1.get(i, None)
    interpretation_output = final_answers_1[i] if i < len(final_answers_1) else None

    conclusions_1[i] = make_conclusion(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        interpretation_code,
        interpretation_output
    )



[2025-12-05 02:18:47.72][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 02:18:52.97][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  5.25 seconds
[2025-12-05 02:18:52.98][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 02:18:53.00][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 02:18:59.16][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  6.16 seconds
[2025-12-05 02:18:59.16][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [38]:
conclusions_2 = {}

for i in range(num_analyses_2):
    independent_variable = features_2[i]['independent_variables']
    dependent_variable = features_2[i]['response_variables']

    model_code = multirun_analyses_2['analyses'][str(i)]['m_code']

    interpretation_code = final_answer_code_2.get(i, None)
    interpretation_output = final_answers_2[i] if i < len(final_answers_2) else None

    conclusions_2[i] = make_conclusion(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        interpretation_code,
        interpretation_output
    )

[2025-12-05 02:02:07.89][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 02:02:15.31][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  7.42 seconds
[2025-12-05 02:02:15.32][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 02:02:15.33][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 02:02:20.48][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  5.15 seconds
[2025-12-05 02:02:20.49][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 02:02:20.50][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 02:02:28.73][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  8.22 

In [39]:
print(conclusions_2)

{0: '{\n  "answer": "Yes",\n  "justification": "The negative-binomial model estimates an IRR = 1.21 for SkinDark (log IRR = 0.194, 95% CI [1.04, 1.42], p = 0.015), indicating dark-skinned players receive red cards at a ~21% higher rate than light-skinned players, controlling for covariates and games."\n}', 1: '{\n  "answer": "Not enough information",\n  "justification": "The model-interpretation output reports that model_output is None and no parameter estimates were extracted. Without the fitted model results (coefficient, p-value, or IRR for Dark vs Light), we cannot determine whether dark-skinned players receive more red cards."\n}', 2: '{\n  "answer": "No",\n  "justification": "The estimated effect is positive (IRR = 1.161) but not statistically significant (coef = 0.149, p = 0.0528; 95% CI for IRR: 0.998–1.350 includes 1). Therefore there is not sufficient evidence that dark-skinned players receive more red cards than light-skinned players."\n}'}


In [21]:
data_head = data.head(10)

In [22]:
print()

In [52]:
results = run_judge_evaluation_pairwise(
    task,
    data_head,
    features_1, features_2,
    model_info_1, model_info_2,
    conclusions_1, conclusions_2,
    llm_provider="openai",
    llm_model="gpt-5-mini",
    output_path="pairwise_results.json"
)


[2025-12-05 02:18:59.23][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/projects/binyu/hao_huang/stat-genie/config/llm_eval_config.yml'.


[2025-12-05 02:18:59.57][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 02:19:12.78][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  13.21 seconds
[2025-12-05 02:19:12.79][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 02:19:12.85][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 02:19:23.77][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  10.92 seconds
[2025-12-05 02:19:23.77][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-05 02:19:23.83][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-05 02:19:37.42][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  13.

In [53]:
results

{(0,
  0): '[Message(role=\'assistant\', content=\'{\\n  "independent_variables": 4,\\n  "control_variables": 3,\\n  "response_variables": 4,\\n  "model_specification": 3,\\n  "conclusions": 1,\\n  "overall_similarity": 2\\n}\')]',
 (0,
  1): '[Message(role=\'assistant\', content=\'{\\n  "independent_variables": 4,\\n  "control_variables": 4,\\n  "response_variables": 5,\\n  "model_specification": 4,\\n  "conclusions": 5,\\n  "overall_similarity": 4\\n}\')]',
 (0,
  2): '[Message(role=\'assistant\', content=\'{\\n  "independent_variables": 4,\\n  "control_variables": 3,\\n  "response_variables": 4,\\n  "model_specification": 4,\\n  "conclusions": 1,\\n  "overall_similarity": 2\\n}\')]',
 (1,
  0): '[Message(role=\'assistant\', content=\'{\\n  "independent_variables": 5,\\n  "control_variables": 4,\\n  "response_variables": 5,\\n  "model_specification": 4,\\n  "conclusions": 5,\\n  "overall_similarity": 5\\n}\')]',
 (1,
  1): '[Message(role=\'assistant\', content=\'{\\n  "independent_va